# Python Plotting & Summary Analysis 

Plots for the jensenRummel2026 paper


Instructions: uses the conda environment via
```bash
cd PATH/TO/Spheroidal3D-collisions/Janus_3D_code/LCPsolvers/jensenRummel2025
conda env create -f environment.yml
```
If you need to update the conda environment file use:
```bash
conda env export --no-builds > environment.yml
```

In [24]:
# Includes
%load_ext autoreload
%autoreload 2
import os, warnings, bson
import numpy as np
from datetime import datetime
# Plotlyjs
import plotly.graph_objects as go
# matplotlib
# %matplotlib inline
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# from matplotlib.ticker import AutoMinorLocator
# File IO -> mat files
import scipy.io

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
BASE_DIR = "/Users/niru8088/scratch/Spheroidal3D-collisions"
FIG_DIR = "/Users/niru8088/scratch/Spheroidal3D-collisions/docs/fig"
RESULTS_DIR = '/Users/niru8088/scratch/Spheroidal3D-collisions/Janus_3D_code/onlineResults'

In [26]:
algo_names = ['BB-PGD', 'Monofidelity PQN', 'B-PQN']
ALGO_TO_DISPLAY = {
    'BB-PGD' : 'BB-PGD', 
    'Monofidelity PQN' : 'Mono-PQN', 
    'B-PQN' : 'Bi-PQN'
}
lattice_size = [3,4,5,6]
lattice_size_2_file_names = {
    3: [    
       'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2_noWarmStart.profile.mat',
       "amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2_noWarmStart.profile.mat",
       'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    ],
    4: [
        'amphi.lattice.n_4.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2_noWarmStart.profile.mat',
        "amphi.lattice.n_4.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2_noWarmStart.profile.mat",
        'amphi.lattice.n_4.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    ],
    5 : [
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.profile.mat'
        ,
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.profile.mat',
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    ],
    6 :[
        "amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.profile.mat",
        "amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.profile.mat",
        'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.profile.mat'
    ]
}

algo_name_to_log_files = {
    'BB-PGD': [
        'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_4.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.diary.log',
        'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2verification.new.diary.log'
    ],
    'Monofidelity PQN': [
        'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_4.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.diary.log',
        'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_proxquasinewton.polyDisperseRatio_0.2verification.new.diary.log',
    ],
    'B-PQN' : [
        'amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_4.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_5.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.diary.log',
        'amphi.lattice.n_6.p_8.cDist_3.lcpSlvr_bifi.polyDisperseRatio_0.2_noWarmStart.diary.log',
    ]
}


## Timing *.mat File Analysis

### Summary: Helper Functions

In [27]:
from datetime import timedelta
def _format_timedelta(td):
    # 1. Extract days directly
    days = td.days
    
    # 2. Extract seconds remaining after days are removed
    total_seconds = td.seconds
    
    # 3. Calculate hours and minutes
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    if days > 0:
        return f"{days}d {hours}h {minutes}m"
    return f"{hours}h {minutes}m"
    

def getFullRunTime(iterTimes, timeData, ixEnd=None):
    Nt = len(iterTimes)
    if ixEnd is None: 
        ixEnd = np.where(iterTimes > 0)[0][-1]
    if ixEnd != Nt-1:
        print(f'--- Run did not finish: {ixEnd=}')
        estimatedTimeSeconds = (timeData['timings']['setup_kernel'][0][0][0] + timeData['timings']['setup_surf'][0][0][0] 
            + np.sum(iterTimes[:ixEnd]) + iterTimes[ixEnd] * (Nt - ixEnd))
        estimatedTimeSeconds = estimatedTimeSeconds[0]
        return estimatedTimeSeconds, _format_timedelta(timedelta(seconds=round(estimatedTimeSeconds/60)*60))
    # If the run finished don't estimate anything
    timeSeconds = timeData['timings']['fullRunTime'][0][0][0]
    timeSeconds = timeSeconds[0]
    return timeSeconds, _format_timedelta(timedelta(seconds=round(timeSeconds/60)*60))


def getCollisionRunTime(colTimesPerIter, ixEnd=None):
    Nt = len(colTimesPerIter)
    if ixEnd is None: 
        ixEnd = np.where(colTimesPerIter > 0)[0][-1]
    if ixEnd != Nt-1:
        print(f'--- Run did not finish: {ixEnd=}')
        estimatedTimeSeconds = np.sum(colTimesPerIter[:ixEnd]) + colTimesPerIter[ixEnd] * (Nt - ixEnd)
        return estimatedTimeSeconds, _format_timedelta(timedelta(seconds=round(estimatedTimeSeconds/60)*60))
    # If the run finished don't estimate anything
    timeSeconds = np.sum(colTimesPerIter)
    return timeSeconds, _format_timedelta(timedelta(seconds=round(timeSeconds/60)*60))

In [28]:
# colors : Cool Gray, Steel Blue, Deep Plum, Slate Blue, Terracotta, 
#          Seafoam Green, Burnt Orange, Mustard Gold Deep Indigo
solid_colors = ["#006450",  "#4664AA", "#78828C", "#2D4178", "#A06428", "#643C73", # 7C - 8b+
                "#DAA520",  "#287882", "#B4503C"] # 8c - 9a
        # ,]
shade_colors_hex = [ "#00645033","#4664AA33","#78828C33", "#2D417833", "#A0642833", "#643C7333", # 7C - 8b+
                 "#DAA52040",  "#28788233", "#B4503C33"] # 8c - 9a 
                # ]
shade_colors = ["rgba(0, 100, 80, 0.2)","rgba(70, 100, 170, 0.2)", "rgba(120, 130, 140, 0.2)","rgba(45, 65, 120, 0.2)","rgba(160, 100, 40, 0.2)", "rgba(100, 60, 115, 0.2)", # 7c - 8b+
                "rgba(218, 165, 32, 0.25)","rgba(40, 120, 130, 0.2)","rgba(180, 80, 60, 0.2)"] # 8c - 9a 
                # ,]
# Neutrals :  Carbon Black, Warm Slate
neutral_solid = ["rgb(40, 40, 40)", "rgb(100, 95, 90)",]
neutral_shade = ["rgba(40, 40, 40, 0.15)","rgba(100, 95, 90, 0.2)"]

In [29]:
def _timing_deep_dive_plot(matrix, cMatrix, algo_names):
    iters = np.arange(start=1, stop=matrix.shape[0]+1)
    ## Plot the timing info per iteration for each algorithm 
    fig = go.Figure()
    ## Loop through the columns of the data matrix and make a line for each algorithm 
    for i, name in zip(range(matrix.shape[1]), algo_names):
        fig.add_trace(go.Scatter(
            x=iters,
            y=matrix[:, i],
            mode='lines',
            name="total",
            legendgroup=ALGO_TO_DISPLAY[name],
            legendgrouptitle=dict(text=name),
            line_color=solid_colors[i],
        ))
        fig.add_trace(go.Scatter(
            x=iters,
            y=cMatrix[:, i],
            mode='lines',
            name="collision",
            legendgroup=ALGO_TO_DISPLAY[name],
            line_color=solid_colors[i],
            line_dash="dash"
        ))

    ## Update the Layout
    fig.update_layout(
        title=dict(
            x=0.5,
            text="<b>Time Per Iteration</b>",
            font=dict(size=22) 
        ),
        xaxis=dict(
            title="Iteration", 
            # tickvals=np.arange(start=0, stop=len(labels)),
            # ticktext=[f'{yr}<br>{name}' for name, yr in zip(labels,climberBestYears)],
            # range=[-0.6, len(labels)-.4],
            # zeroline=False,
            showgrid=False,
            tickfont=dict(size=14),   
            title_font=dict(size=18)
        ),
        yaxis=dict(
            title='<b>Time (s)</b>',
            # tickvals=bin_centers,
            # ticktext=[SCORE_TO_FONT.get(int(t), t) for t in bin_centers],
            gridcolor='lightgrey',
            tickfont=dict(size=14),   
            title_font=dict(size=18)
        ),
        legend=dict(
            title=dict(
                text="Total Time Per Iteration",
                font_size=18
            ),
            orientation="v",      # Vertical orientation is standard for side legends
            yanchor="middle",     # Centers the legend vertically relative to the 'y' value
            y=0.5,                # Places legend at the vertical midpoint (50%)
            xanchor="left",       # Anchors the left side of the legend to the 'x' coordinate
            x=1.02,               # Moves it slightly to the right of the plot area
            bgcolor='rgba(255, 255, 255, 0.5)', # Optional: semi-transparent background
            bordercolor="Black",
            borderwidth=1
        ),
        font=dict(
            family="Arial, sans-serif", # Clean, web-safe font
            size=15,                   # Global default size
            color="black"
        ),
        hovermode='x unified',
        barmode='overlay', # Crucial for overlapping All/FA/Flash
        plot_bgcolor='white',
        # Increase the right margin so the legend doesn't get cut off
        margin=dict(r=150),
        height=600,
        autosize=True
    )

    config = {'responsive': True}
    fig.show(config=config)

## Save to pdf file
# fig.write_image("totalTiming.pdf", width=800, height=600, scale=2)

## Save the figure as a standalone interactive HTML file
# html_file = '.html'
# height_script = """
#     var sendHeight = function() {
#         var height = document.body.scrollHeight;
#         window.parent.postMessage({ 'height': height }, "*");
#     };
#     window.onload = sendHeight;
#     // Also send height if the window is resized
#     window.onresize = sendHeight;
# """
# fig.write_html(os.path.join(FIG_DIR, html_file), full_html=True, include_plotlyjs='cdn', config=config, post_script=height_script)

In [30]:

def timing_deep_dive(file_names, algo_names, plotFlag=False):
    assert len(file_names) == len(algo_names)
    timeData = [scipy.io.loadmat(os.path.join(RESULTS_DIR,fn)) for fn in file_names]
    # Create a matrix where each column is the total time for each iteration of each algorithm
    matrix = np.hstack([td['timings']['total'][0][0] for td in timeData])
    cMatrix = np.hstack([td['timings']['velocities'][0][0]['col'][0][0] for td in timeData])
    if plotFlag: _timing_deep_dive_plot(matrix, cMatrix, algo_names)

    ##
    total_run_times = []
    total_speed_up = []
    print(f'Total Run Time')
    baseline_seconds, baseline_duration = getFullRunTime(matrix[:,0], timeData[0])
    total_run_times.append(baseline_duration)
    total_speed_up.append("NA")
    print(f'- {algo_names[0]} {baseline_duration}')
    for i, an in enumerate(algo_names[1:]):
        seconds, duration =  getFullRunTime(matrix[:,i+1], timeData[i+1])
        print(f'- {an} {duration}')
        speedUp = baseline_seconds / seconds
        print(f'-- {an} is {speedUp:.2f} faster than {algo_names[0]}')
        total_run_times.append(duration)
        total_speed_up.append(speedUp)

    ## 
    print(f'Collision Resolution Run Time')
    baseline_seconds, baseline_duration =  getCollisionRunTime(cMatrix[:,0])
    collision_run_times = []
    collision_speed_up = []
    collision_run_times.append(baseline_duration)
    collision_speed_up.append("NA")
    print(f'- {algo_names[0]} {baseline_duration}')
    for i, an in enumerate(algo_names[1:]):
        seconds, duration =  getCollisionRunTime(cMatrix[:,i+1])
        print(f'- {an} {duration}')
        speedUp = baseline_seconds / seconds
        print(f'-- {an} is {speedUp:.2f} faster than {algo_names[0]}')
        collision_run_times.append(duration)
        collision_speed_up.append(speedUp)

    return total_run_times, total_speed_up, collision_run_times, collision_speed_up

    

### Print Summary

In [31]:
total_run_times = []
total_speed_up = []
collision_run_times = []
collision_speed_up = []
for n in lattice_size:
    print('=================================================')
    print(f'{n=}')
    _total_run_times, _total_speed_up, _collision_run_times, _collision_speed_up = timing_deep_dive(lattice_size_2_file_names[n], algo_names, plotFlag=True)
    total_run_times.append(_total_run_times)
    total_speed_up.append(_total_speed_up)
    collision_run_times.append(_collision_run_times)
    collision_speed_up.append(_collision_speed_up)

n=3


Total Run Time
- BB-PGD 15h 21m
- Monofidelity PQN 9h 41m
-- Monofidelity PQN is 1.59 faster than BB-PGD
- B-PQN 8h 18m
-- B-PQN is 1.85 faster than BB-PGD
Collision Resolution Run Time
- BB-PGD 13h 31m
- Monofidelity PQN 7h 58m
-- Monofidelity PQN is 1.70 faster than BB-PGD
- B-PQN 6h 33m
-- B-PQN is 2.07 faster than BB-PGD
n=4


Total Run Time
- BB-PGD 1d 3h 54m
- Monofidelity PQN 20h 23m
-- Monofidelity PQN is 1.37 faster than BB-PGD
- B-PQN 15h 54m
-- B-PQN is 1.75 faster than BB-PGD
Collision Resolution Run Time
- BB-PGD 23h 26m
- Monofidelity PQN 15h 44m
-- Monofidelity PQN is 1.49 faster than BB-PGD
- B-PQN 11h 11m
-- B-PQN is 2.10 faster than BB-PGD
n=5


Total Run Time
- BB-PGD 2d 18h 36m
- Monofidelity PQN 2d 2h 26m
-- Monofidelity PQN is 1.32 faster than BB-PGD
- B-PQN 1d 17h 40m
-- B-PQN is 1.60 faster than BB-PGD
Collision Resolution Run Time
- BB-PGD 2d 0h 43m
- Monofidelity PQN 1d 10h 10m
-- Monofidelity PQN is 1.43 faster than BB-PGD
- B-PQN 23h 12m
-- B-PQN is 2.10 faster than BB-PGD
n=6


Total Run Time
--- Run did not finish: ixEnd=np.int64(183)
- BB-PGD 7d 18h 0m
- Monofidelity PQN 6d 16h 16m
-- Monofidelity PQN is 1.16 faster than BB-PGD
- B-PQN 4d 22h 44m
-- B-PQN is 1.57 faster than BB-PGD
Collision Resolution Run Time
--- Run did not finish: ixEnd=np.int64(183)
- BB-PGD 5d 0h 25m
- Monofidelity PQN 3d 19h 27m
-- Monofidelity PQN is 1.32 faster than BB-PGD
- B-PQN 2d 0h 50m
-- B-PQN is 2.47 faster than BB-PGD


In [32]:
total_speed_up

[['NA', np.float64(1.5855707932206158), np.float64(1.8484041577990074)],
 ['NA', np.float64(1.368583534357436), np.float64(1.7538638950988237)],
 ['NA', np.float64(1.3202204578002605), np.float64(1.598086155408705)],
 ['NA', np.float64(1.1605775957921995), np.float64(1.5666829925590628)]]

In [33]:
from pylatex import Table, Tabular, LargeText, HugeText,Command
from pylatex.utils import bold, NoEscape
# This allows you to build the table programmatically row-by-row
# table = Table()
spaceCmd = Command('hspace','.25cm').dumps()
tabular = Tabular('lcccc')
tabular.append(NoEscape(r'\toprule'))
tabular.add_row([""]+[f'{n}' for n in lattice_size])
tabular.append(NoEscape(r'\midrule'))
for i,an in enumerate(algo_names):
    tabular.add_row([bold(ALGO_TO_DISPLAY[an])]+['' for _ in total_run_times])
    if i == 0:
        tabular.add_row([
            NoEscape(f'{spaceCmd} Total Run Time')]+[rt[i] for rt in total_run_times])
        tabular.add_row([
            NoEscape(f'{spaceCmd} Collision Resolution')]+[rt[i] for rt in collision_run_times])
    else:
        tabular.add_row([
            NoEscape(f'{spaceCmd} Total Run Time')]+[NoEscape('${:.2f}\\times$'.format(su[i])) for su in total_speed_up])
        tabular.add_row([
            NoEscape(f'{spaceCmd} Collision Resolution')]+[NoEscape('${:.2f}\\times$'.format(su[i])) for su in collision_speed_up])
tabular.append(NoEscape(r'\bottomrule'))
with open(os.path.join(FIG_DIR, 'heroRun_table.tex'),'w+') as wf: 
    wf.write(tabular.dumps())

In [34]:
Command('hspace','.25cm')

Command('hspace', Arguments('.25cm'), Options())

## Log File Scraping

In [35]:
def scrape_log(fn):
    with open(os.path.join(RESULTS_DIR, fn), 'r') as file:
        # .readlines() creates a list where each element is one line from the file
        lines = file.readlines()

    # Stores tuples of (line_number, line_content)
    indices = [(i,line) for i, line in enumerate(lines) if 'Total computing time' in line]
    startIx = 0
    lines_per_timestep = []
    for i, content in indices:
        # print(f"Line {i}: {content.rstrip()}")
        # print(lines[i])
        lines_per_timestep.append(lines[startIx:i])
        startIx = i
    ## Collect summary stats 
    lcp_sizes = []
    mvp_times = []
    lofi_mvp_times = []
    mvp_cnts = []
    emvp_cnts = []
    lofi_mvp_cnts = []
    lcp_solve_times = []
    lcp_solve_tol = []

    for lines in lines_per_timestep:
        try: 
            ## Calculate the size of the lcp
            ix_and_line_before_sz = [(i,line) for i, line in enumerate(lines) if 'Time for contact force correction' in line][0]
            lines_with_cols = [line for line in lines[:ix_and_line_before_sz[0]] if 'Columns' in line]
            n = int(lines_with_cols[-1].split(' ')[-1])
            print(f'- {n=}')
            ## Compute mvp info
            lines_with_MVP = [(i,line) for i, line in enumerate(lines) if 'A[x] MVP time' in line]
            lines_with_lofi_MVP = [(i,line) for i, line in enumerate(lines) if 'Ahat[x] MVP time' in line]
            _mvp_times = []
            for i, line in lines_with_MVP:
                # print(f'- {line=}')
                parts = line.split('time ')
                parts = parts[-1].split(' sec')
                _mvp_times.append(float(parts[0].strip()))
            print(f'- {_mvp_times=}')
            _lofi_mvp_times = []
            for i, line in lines_with_lofi_MVP:
                # print(f'- {line=}')
                parts = line.split('time ')
                parts = parts[-1].split(' sec')
                _lofi_mvp_times.append(float(parts[0].strip()))
            print(f"- {_lofi_mvp_times=}")
            num_mvps = len(lines_with_MVP)
            num_emvps = num_mvps
            num_lofi_mvps = len(lines_with_lofi_MVP)
            if num_lofi_mvps > 0 :
                fctr =  np.array(_lofi_mvp_times).mean() / np.array(_mvp_times).mean() # adjust for the relative cost of Ahat vs A
                num_emvps += num_lofi_mvps * fctr
            ix_and_line_with_LCP_sol = [(i,line) for i, line in enumerate(lines) if 'LCP solution error' in line][0]

            parts = ix_and_line_with_LCP_sol[1].split('=')
            lcp_error = float(parts[1].split(',')[0].strip())
            lcp_iters = int(parts[2].strip())
            line_with_solve_time = lines[ix_and_line_with_LCP_sol[0]+2]
            parts = line_with_solve_time.split('=')
            lcp_solve_time = float(parts[1].strip())
            print(f'- {lcp_solve_time=}')

            lcp_sizes.append(n)
            mvp_times.append(_mvp_times)
            lofi_mvp_times.append(_lofi_mvp_times)
            mvp_cnts.append(num_mvps)
            emvp_cnts.append(num_emvps)
            lofi_mvp_cnts.append(num_lofi_mvps)
            lcp_solve_times.append(lcp_solve_time)
            lcp_solve_tol.append(lcp_error)
        except:
            pass
    return {
        'lcp_sizes': lcp_sizes,
        'mvp_times': mvp_times,
        'lofi_mvp_times': lofi_mvp_times,
        'mvp_cnts': mvp_cnts,
        'emvp_cnts': emvp_cnts,
        'lofi_mvp_cnts': lofi_mvp_cnts,
        'lcp_solve_times': lcp_solve_times,
        'lcp_solve_tol': lcp_solve_tol,
    }

In [36]:
retDic = {}
for algo, file_names in algo_name_to_log_files.items():
    print('==============================================')
    retDic[algo] = {
        'lcp_sizes': [],
        'mvp_times': [],
        'lofi_mvp_times': [],
        'mvp_cnts': [],
        'emvp_cnts': [],
        'lofi_mvp_cnts': [],
        'lcp_solve_times': [],
        'lcp_solve_tol': []
    }
    for fn in file_names:
        print(f'{algo} : {fn}')
        _thisRetDic = scrape_log(fn)
        retDic[algo]['lcp_sizes'] += _thisRetDic['lcp_sizes']
        retDic[algo]['mvp_times'] += _thisRetDic['mvp_times']
        retDic[algo]['lofi_mvp_times'] += _thisRetDic['lofi_mvp_times']
        retDic[algo]['mvp_cnts'] += _thisRetDic['mvp_cnts']
        retDic[algo]['emvp_cnts'] += _thisRetDic['emvp_cnts']
        retDic[algo]['lofi_mvp_cnts'] += _thisRetDic['lofi_mvp_cnts']
        retDic[algo]['lcp_solve_times'] += _thisRetDic['lcp_solve_times']
        retDic[algo]['lcp_solve_tol'] += _thisRetDic['lcp_solve_tol']

BB-PGD : amphi.lattice.n_3.p_8.cDist_3.lcpSlvr_bbpgd.polyDisperseRatio_0.2_noWarmStart.diary.log
- n=27
- _mvp_times=[27.6, 26.2, 25.1, 24.5, 26.2, 23.5, 24.5, 24.3, 23.7, 23.1, 23.5, 25.8]
- _lofi_mvp_times=[]
- lcp_solve_time=297.928
- n=27
- _mvp_times=[92.0, 97.7, 80.8, 95.5, 90.1, 86.9, 79.5, 73.4, 57.9, 95.4, 75.1]
- _lofi_mvp_times=[]
- lcp_solve_time=924.493
- n=27
- _mvp_times=[28.9, 29.1, 31.4, 28.1, 29.0, 29.5, 28.5, 27.5, 27.4, 24.6, 25.6]
- _lofi_mvp_times=[]
- lcp_solve_time=309.499
- n=27
- _mvp_times=[20.7, 20.8, 22.2, 20.2, 21.3, 20.3]
- _lofi_mvp_times=[]
- lcp_solve_time=125.634
- n=27
- _mvp_times=[20.7, 20.8, 20.8, 21.3, 22.3]
- _lofi_mvp_times=[]
- lcp_solve_time=105.982
- n=27
- _mvp_times=[20.7, 21.2, 21.5, 21.4, 21.4, 21.4]
- _lofi_mvp_times=[]
- lcp_solve_time=127.667
- n=27
- _mvp_times=[20.7, 21.4, 21.2, 21.3, 22.8, 21.1]
- _lofi_mvp_times=[]
- lcp_solve_time=128.547
- n=27
- _mvp_times=[21.1, 21.1, 21.4, 21.2, 23.0, 21.2]
- _lofi_mvp_times=[]
- lcp_solve_ti

### Plot

In [37]:
name2Color = {
    "BB-PGD": "#1f77b4",  # muted blue
    "Monofidelity PQN": "#9467bd",  # muted purple
    "B-PQN": "#CFB87C",  # CU Gold
}
name2FillColor = {
    "BB-PGD":"rgba(31, 119, 180, 0.5)",
    "Monofidelity PQN":"rgba(148, 103, 189, 0.5)",
    "B-PQN":"rgba(207, 184, 124, 0.5)"
}
algo_names = list(name2Color.keys())

In [38]:
# from plotly.subplots import make_subplots
# fig = make_subplots(
#     rows=len(algo_names), cols=1, 
#     shared_xaxes=True, 
#     vertical_spacing=0.08,
#     subplot_titles=algo_names
# )
fig = go.Figure()
metric_name = 'emvp_cnts'

for ai, algo_name in enumerate(algo_names):
    x_data = retDic[algo_name]['lcp_sizes']
    y_data = retDic[algo_name][metric_name]
    unique_x = list(set(x_data))
    unique_x.sort()
    ix_for_each_x = [[i for i,xi in enumerate(x_data) if xi == xu] for xu in unique_x]
    y_for_each_x = []
    for ix in ix_for_each_x:
        y_for_each_x.append([y_data[i] for i in ix])
    for i, (xu,yu) in enumerate(zip(lattice_size, y_for_each_x)): 
        fig.add_trace(
            go.Box(
                x=[xu + .3*(ai - 1) for _ in yu],
                y=yu,
                line_color=name2Color[algo_name],
                fillcolor=name2FillColor[algo_name],
                # opacity=0.6,
                showlegend=i==0,
                legendgroup=algo_name,
                name=ALGO_TO_DISPLAY[algo_name]
            ),
            # row=ai+1, col=1
        )
fig.update_xaxes(
    tickmode='array',
    tickvals=lattice_size,
    # ticktext=['10 Units', '50 Units', '100 Units', '500 Units (Max)']
)
fig.update_layout(
    font=dict(
        family="Arial, sans-serif",
        size=18, # Increasing this from the default (12)
        # color="#2C3E50"
    ),
    title=dict(
        # text= "<b>Distribution of Effective MVPs vs Problem Size</b>",
        y= 0.95,           # Vertical position
        x= 0.5,            # Centered horizontally
        xanchor= 'center',
        yanchor= 'top',
        font= dict(
            family="Arial, sans-serif",
            size=24, 
            # color='#2C3E50'
        )
    ),
    legend=dict(
        title=dict(
            text = "<b>LCP Solvers</b>", # Legend Title
            side = 'top center',
            font = dict(
                family="Arial, sans-serif",
                size=20
            )
        ),
        bgcolor = "#F2F4F4",              # Same Gray Shade
        bordercolor = "#BDC3C7",          # Darker Gray Border
        borderwidth = 2,
        x = .85,                         # Position inside the plot
        y = 1.1,
        xanchor = 'left',
        yanchor = 'top'
    ),
    xaxis=dict(
        title=dict(
            text="<b>Lattice Size</b>",
            font=dict(
                family="Arial, sans-serif",
                size=20
            )
        )
    ),
    yaxis=dict(
        domain=[0,1],
        title=dict(
            text="<b>E-MVPs</b>",
            font=dict(
                family="Arial, sans-serif",
                size=20
            )
        )
    ),
    template="plotly_white",
    # violinmode='group'
    width=900, 
    height=500,
    margin=dict(t=60, b=60, l=60, r=60)
)

fig.write_image(os.path.join(FIG_DIR, "heroRuns_boxPlots.pdf"), scale=2)
fig.show()